<a href="https://colab.research.google.com/github/Ashish-kharde1/credit-card-fraud-detection/blob/main/credit_card_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [65]:
!pip install lightgbm imbalanced-learn shap

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.metrics import precision_score, recall_score, fbeta_score, roc_auc_score, matthews_corrcoef, classification_report
import pickle

df = pd.read_csv('fraudTrain.csv')
df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Unnamed: 0             2001 non-null   int64  
 1   trans_date_trans_time  2001 non-null   object 
 2   cc_num                 2001 non-null   int64  
 3   merchant               2001 non-null   object 
 4   category               2001 non-null   object 
 5   amt                    2001 non-null   float64
 6   first                  2001 non-null   object 
 7   last                   2001 non-null   object 
 8   gender                 2001 non-null   object 
 9   street                 2001 non-null   object 
 10  city                   2001 non-null   object 
 11  state                  2001 non-null   object 
 12  zip                    2001 non-null   int64  
 13  lat                    2001 non-null   float64
 14  long                   2001 non-null   float64
 15  city

In [68]:
df.describe()

,Unnamed: 0,cc_num,amt,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,is_fraud
count,2001.000000,2.001000e+03,2001.000000,2001.000000,2001.000000,2001.000000,2.001000e+03,2.001000e+03,2001.000000,2001.000000,2001.0
mean,1000.000000,3.963592e+17,67.299845,49707.683158,38.572305,-90.675702,8.491219e+04,1.325417e+09,38.577782,-90.680655,0.0
std,577.783264,1.274926e+18,120.271803,27030.757753,5.025286,14.280923,2.707642e+05,2.128676e+04,5.065120,14.289091,0.0
min,0.000000,6.041621e+10,1.030000,1257.000000,20.027100,-165.672300,2.300000e+01,1.325376e+09,19.209212,-166.148374,0.0
25%,500.000000,1.800365e+14,9.300000,25832.000000,35.058300,-97.289300,7.600000e+02,1.325400e+09,35.083586,-97.418265,0.0
50%,1000.000000,3.519233e+15,47.960000,49629.000000,39.390000,-88.043400,2.443000e+03,1.325422e+09,39.355193,-88.129609,0.0
75%,1500.000000,4.658491e+15,79.900000,72341.000000,41.698300,-80.158000,1.909000e+04,1.325435e+09,41.826473,-80.286355,0.0
max,2000.000000,4.992346e+18,3178.510000,99783.000000,64.755600,-67.950300,2.906700e+06,1.325450e+09,65.389987,-67.938911,0.0


In [69]:
df.is_fraud.value_counts()

is_fraud
0    2001
Name: count, dtype: int64

In [70]:
df.columns

Index(['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category',
       'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip',
       'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time',
       'merch_lat', 'merch_long', 'is_fraud'],
      dtype='object')

In [71]:
df.drop(["cc_num","merchant","first","last","street","zip","trans_num","Unnamed: 0"], axis=1, inplace=True)

In [72]:
df.columns

Index(['trans_date_trans_time', 'category', 'amt', 'gender', 'city', 'state',
       'lat', 'long', 'city_pop', 'job', 'dob', 'unix_time', 'merch_lat',
       'merch_long', 'is_fraud'],
      dtype='object')

In [73]:
df.dropna(inplace=True)

In [74]:
df.head()

,trans_date_trans_time,category,amt,gender,city,state,lat,long,city_pop,job,dob,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,misc_net,4.97,F,Moravian Falls,NC,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,grocery_pos,107.23,F,Orient,WA,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,entertainment,220.11,M,Malad City,ID,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,1325376051,43.150704,-112.154481,0
3,2019-01-01 00:01:16,gas_transport,45.00,M,Boulder,MT,46.2306,-112.1138,1939,Patent attorney,1967-01-12,1325376076,47.034331,-112.561071,0
4,2019-01-01 00:03:06,misc_pos,41.96,M,Doe Hill,VA,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,1325376186,38.674999,-78.632459,0


In [75]:
df.category.value_counts()

category
gas_transport     243
grocery_pos       228
shopping_pos      176
home              168
kids_pets         156
misc_pos          153
shopping_net      146
personal_care     138
food_dining       128
health_fitness    115
entertainment     110
misc_net          107
grocery_net        79
travel             54
Name: count, dtype: int64

In [76]:
df['dob'] = pd.to_datetime(df['dob'], errors='coerce')
df['age'] = (pd.Timestamp.now() - df['dob']).dt.days // 365
df.drop('dob', axis=1, inplace=True)

In [77]:
def haversine(lat1, lon1, lat2, lon2):
  R = 6371 #Earth radius in Km
  lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
  dlat = lat2 - lat1
  dlon = lon2 - lon1
  a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
  return 2 * R * np.arcsin(np.sqrt(a))

df['distance'] = haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])

In [78]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'], format='%Y-%m-%d %H:%M:%S')

df['hour'] = df['trans_date_trans_time'].dt.hour
df['day'] = df['trans_date_trans_time'].dt.day
df['month'] = df['trans_date_trans_time'].dt.month
df['weekday'] = df['trans_date_trans_time'].dt.weekday

df.drop('trans_date_trans_time', axis=1, inplace=True)

In [79]:
cat_cols = ['category','gender','city','state','job']
encoder = LabelEncoder()
df["category"] = encoder.fit_transform(df["category"])
df["gender"] = encoder.fit_transform(df["gender"])
df["city"] = encoder.fit_transform(df["city"])
df["state"] = encoder.fit_transform(df["state"])
df["job"] = encoder.fit_transform(df["job"])

In [80]:
df.columns

Index(['category', 'amt', 'gender', 'city', 'state', 'lat', 'long', 'city_pop',
       'job', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'age',
       'distance', 'hour', 'day', 'month', 'weekday'],
      dtype='object')

In [81]:
df.head()

,category,amt,gender,city,state,lat,long,city_pop,job,unix_time,merch_lat,merch_long,is_fraud,age,distance,hour,day,month,weekday
0,8,4.97,0,417,26,36.0788,-81.1781,3495,328,1325376018,36.011293,-82.048315,0,38,78.597568,0,1,1,1
1,4,107.23,0,479,46,48.8878,-118.2105,149,380,1325376044,49.159047,-118.186462,0,48,30.212176,0,1,1,1
2,0,220.11,1,372,12,42.1808,-112.2620,4154,268,1325376051,43.150704,-112.154481,0,64,108.206083,0,1,1,1
3,2,45.00,1,69,25,46.2306,-112.1138,1939,288,1325376076,47.034331,-112.561071,0,59,95.673231,0,1,1,1
4,9,41.96,1,176,44,38.4207,-79.4629,99,96,1325376186,38.674999,-78.632459,0,40,77.556744,0,1,1,1


In [82]:
target = 'is_fraud'
num_features = [
    'amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'age', 'distance',
    'hour', 'day', 'month', 'weekday'
]
cat_features = [
    'category', 'gender', 'state', 'city', 'job'
]

In [83]:
num_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ])

In [84]:
df.head()

,category,amt,gender,city,state,lat,long,city_pop,job,unix_time,merch_lat,merch_long,is_fraud,age,distance,hour,day,month,weekday
0,8,4.97,0,417,26,36.0788,-81.1781,3495,328,1325376018,36.011293,-82.048315,0,38,78.597568,0,1,1,1
1,4,107.23,0,479,46,48.8878,-118.2105,149,380,1325376044,49.159047,-118.186462,0,48,30.212176,0,1,1,1
2,0,220.11,1,372,12,42.1808,-112.2620,4154,268,1325376051,43.150704,-112.154481,0,64,108.206083,0,1,1,1
3,2,45.00,1,69,25,46.2306,-112.1138,1939,288,1325376076,47.034331,-112.561071,0,59,95.673231,0,1,1,1
4,9,41.96,1,176,44,38.4207,-79.4629,99,96,1325376186,38.674999,-78.632459,0,40,77.556744,0,1,1,1


In [85]:
X = df.drop(target, axis=1)
y = df[target]
X_preprocessed = preprocessor.fit_transform(X)

In [86]:
X.columns

Index(['category', 'amt', 'gender', 'city', 'state', 'lat', 'long', 'city_pop',
       'job', 'unix_time', 'merch_lat', 'merch_long', 'age', 'distance',
       'hour', 'day', 'month', 'weekday'],
      dtype='object')

In [87]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

model = lgb.LGBMClassifier()

In [88]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [89]:
params = {
    "objective": "binary",
    "boosting_type": "gbdt",
    "learning_rate": 0.03,  # Decreased learning rate
    "num_leaves": 20,       # Reduced number of leaves
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.3, # Increased L1 regularization
    "lambda_l2": 0.3, # Increased L2 regularization
    "metric": ["auc"],
    "verbosity": -1,
    "is_unbalance": True,   # important for fraud detection
    "min_child_samples": 25 # Increased minimum child samples
}

In [90]:
from lightgbm import early_stopping, log_evaluation

model = lgb.train(
    params,
    train_data,
    num_boost_round=2000,  # Increased number of boosting rounds
    valid_sets=[train_data, val_data],
    valid_names=["train","val"],
    callbacks=[early_stopping(100), log_evaluation(100)] # Increased early stopping rounds
)

Training until validation scores don't improve for 100 rounds
[100]	train's auc: 1	val's auc: 1
Early stopping, best iteration is:
[1]	train's auc: 1	val's auc: 1


In [91]:
from sklearn.metrics import precision_score, recall_score, fbeta_score, roc_auc_score, matthews_corrcoef, classification_report

# If you trained with lgb.train()
y_pred_proba = model.predict(X_test, num_iteration=model.best_iteration)  # gives probability of fraud (class=1)
y_pred = (y_pred_proba > 0.5).astype(int)  # threshold at 0.5



print("AUC:", roc_auc_score(y_test, y_pred_proba))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F2-score:", fbeta_score(y_test, y_pred, beta=2))
print("MCC:", matthews_corrcoef(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

AUC: nan
Precision: 0.0
Recall: 0.0
F2-score: 0.0
MCC: 0.0

Classification Report:
               precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       401

    accuracy                         1.0000       401
   macro avg     1.0000    1.0000    1.0000       401
weighted avg     1.0000    1.0000    1.0000       401



/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted sa

In [92]:

# Save the model
with open('lgb_model.pkl', 'wb') as f:
    pickle.dump(model, f)